In [43]:
import os
import json, re
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [ ]:
# Initialize and constants

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GEMINI_API_KEY")

if not google_api_key:
     print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith("AIz"):
     print("An API key was found, but it doesn't start AIz")
else:
     print("API key found and looks good so far!")
    
MODEL = 'gemini-2.5-flash'
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

In [37]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a marketing content about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You MUST respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a marketing content about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [54]:
def select_relevant_links(url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [ ]:
select_relevant_links("https://huggingface.co/")

In [58]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://huggingface.co/")

In [74]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
fetch_page_and_all_relevant_links("https://huggingface.co/")

In [70]:
marketing_system_prompt = """
You are a Marketing Content Intelligence Engine.

Your job is to analyze content from multiple company pages (website pages, product pages, landing pages, about pages, social posts, etc.) and extract:

1. Brand voice & tone
2. Unique selling points (USPs)
3. Target audience
4. Core problems the company solves
5. Main services or products
6. Emotional triggers and benefits
7. Keywords and repeated phrases
8. Brand positioning vs competitors

Then you must generate high-converting marketing content that:
- Matches the brand's tone
- Uses their real language style
- Focuses on customer benefits (not features)
- Sounds natural, not AI-written
- Is optimized for social media, ads, and landing pages

You must never hallucinate facts.  
Only use information that appears in the provided pages.

Output must always be structured and clear.
"""

In [75]:
def get_content_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short marketing content of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [76]:
def create_marketing_content(company_name, url):
    response = gemini.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[
            {"role": "system", "content": marketing_system_prompt},
            {"role": "user", "content": get_content_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_marketing_content("huggingface","https://huggingface.co/")